# 🎯 SQL: Advanced Window Functions

**Advanced SQL Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** ROW_NUMBER vs RANK, Time-series analysis, NTILE, framing
- **Tags:** `sql` | `window-functions` | `analytics` | `citi-prep`

## 📖 The Core Concept in Plain Language

Standard `GROUP BY` aggregates data into a single row per group (losing the individual row details). **Window Functions** apply an aggregate or ranking over a defined "window" of rows, but they append the result to *every individual row* without collapsing the dataset.

### Why This Matters

- **Preservation of detail:** You can see the individual Employee's exact salary alongside their Department's Average salary on the very same row.
- **Sequential logic:** You can compare "this row" to "the previous row" seamlessly (like calculating daily delta changes in stock prices).
- **Interview Checkpoint:** If you don't know window functions, you won't pass a Data Engineering SQL interview.

### The Key Insight

```sql
SELECT emp_name, salary,
  -- Look at the window of all people in my department, and average it
  AVG(salary) OVER(PARTITION BY department) as dept_avg_salary
FROM employees;
```

## 🔍 The Big Three: Row Number vs Rank vs Dense Rank

In [ ]:
# Demonstrating how ties are handled
print("Imagine scores: 100, 100, 90, 80")
print("ROW_NUMBER(): 1, 2, 3, 4 (Strictly sequential, ties broken arbitrarily)")
print("RANK():       1, 1, 3, 4 (Ties get same rank, but leaves a gap for rank 2)")
print("DENSE_RANK(): 1, 1, 2, 3 (Ties get same rank, NO GAPS. The next best score is rank 2)")

## 🏗️ The Anatomy of Frames (Moving Averages)

When you use `ORDER BY` inside an `OVER()` clause, the database implicitly adds a "framing" clause: `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. This calculates a running total by default.

You can override this frame to create things like a 3-day rolling average.

### Structure:
```sql
AVG(daily_sales) OVER (
    PARTITION BY store_id 
    ORDER BY date_id 
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
) AS rolling_3_day_avg
```

In [ ]:
print("Framing keywords:")
print("- UNBOUNDED PRECEDING (From the start of the partition)")
print("- N PRECEDING (N rows before)")
print("- CURRENT ROW (Exactly here)")
print("- N FOLLOWING (N rows after)")
print("- UNBOUNDED FOLLOWING (To the end of the partition)")

## 🔄 Lag, Lead, and the "Change" Metric

How do you calculate Day-over-Day growth in pure SQL? You use `LAG()`.

In [ ]:
sql_lag = """
SELECT 
    date,
    revenue,
    LAG(revenue, 1) OVER (ORDER BY date) as previous_day_revenue,
    revenue - LAG(revenue, 1) OVER (ORDER BY date) as daily_delta
FROM sales;
"""
print("LAG(col, offset): Reaches back N rows and grabs that value.")
print("LEAD(col, offset): Reaches forward N rows and grabs that value.")
print("Crucial: Without an ORDER BY in the OVER() clause, LAG/LEAD throw errors or return unpredictable garbage.")

## 🌍 Real-World Example: Finding the "Latest" Record

**The Citi Context:** Databases often receive multiple updates for a single entity (like a customer's address changing). The raw table has 5 rows for Customer 123. The analyst relies on you to give them a view containing *only the most recent address*.

### The Window Function Hero (Deduplication pattern)
```sql
WITH ranked_addresses AS (
    SELECT 
        customer_id,
        address,
        updated_at,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) as rn
    FROM customer_history
)
SELECT customer_id, address
FROM ranked_addresses
WHERE rn = 1;
```

In [ ]:
print("This 'rn = 1' CTE pattern is used in almost every staging-to-production ETL pipeline in existence to extract the 'current state'.")

## 🎭 Advanced Distribution: NTILE

`NTILE(x)` forces your ordered results into `x` equally sized buckets. 

In [ ]:
print("Use Case: Identifying Top Quartile performers.")
print("SELECT employee_name, NTILE(4) OVER (ORDER BY sales DESC) as quartile FROM sales_team;")
print("Quartile 1 = Top 25% of salespeople.")

## 🎤 5 Interview Q&A

### Q1: What is the difference between `GROUP BY` and an `OVER(PARTITION BY)` window function?
**Answer:** `GROUP BY` reduces the output. Ten rows in a department become one single row representing the department average. Window functions evaluate the context of the department but preserve all ten original rows, tacking the average onto each row.

---

### Q2: What's the difference between `RANK` and `DENSE_RANK`?
**Answer:** How they handle the number *after* a tie. If scores are 100, 100, 90: `RANK` returns 1, 1, 3. `DENSE_RANK` returns 1, 1, 2. `DENSE_RANK` leaves no gaps.

---

### Q3: You have a table with duplicates. How do you delete the duplicates leaving only one copy?
**Answer:** Use a CTE with `ROW_NUMBER() OVER (PARTITION BY all_duplicate_columns ORDER BY metadata_like_timestamp)`. Then `DELETE FROM cte_name WHERE rn > 1`. (Note: Syntax for deleting from a CTE varies by RDBMS, but the concept is universal).

---

### Q4: If you use `SUM(cost) OVER(PARTITION BY region ORDER BY date)`, what exactly is being calculated?
**Answer:** Because an `ORDER BY` is included, it implicitly creates a window frame from the start of the partition up to the current row. Therefore, it calculates a **running total** (cumulative sum) of costs within that region over time.

---

### Q5: Can you use a Window Function in a `WHERE` clause?
**Answer:** No. Window functions are evaluated last in the SQL logical order of operations (after `WHERE`, `GROUP BY`, and `HAVING`). To filter on the result of a window function, you must put the window function in a CTE or Subquery, and then filter in the outer query.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **Partition By** | Defines the independent groups over which the function operates | `OVER(PARTITION BY dept)` |
| **Order By (Inside OVER)** | Defines the logical sequence of rows within the partition | `OVER(ORDER BY score DESC)` |
| **Frame** | The subset of the partition evaluated for the current row | `ROWS BETWEEN 1 PRECEDING...` |
| **LAG** | Fetch value from a previous row in the partition sequence | `LAG(price, 1)` |
| **NTILE(N)** | Distributes rows into N sequentially numbered buckets | `NTILE(10) OVER(...)` |

## 💼 The Citi Narrative: Deciphering the Last State

### The Problem
At Citi, we tracked agent health telemetry. A server might check in "Healthy" at 9:00, "Warning" at 9:15, and "Offline" at 9:45. Executive reporting only cared about: *"What is the final state of each server at the end of the 24-hour cycle?"*

### The Solution
I couldn't just `GROUP BY server_id` because "Offline" isn't an aggregatable number like a SUM or MAX. I used the `ROW_NUMBER() OVER(PARTITION BY server_id ORDER BY timestamp DESC)` deduplication pattern.

### The Impact
By selecting `rn = 1`, I extracted a perfectly clear 'current state' table for the reporting layer, isolating millions of telemetry pings into an exact 6,000-row inventory representing the final known status globally.

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: First Value
# How do you get the FIRST salary ever recorded for an employee, 
# placed on every row of their history?
print("FIRST_VALUE(salary) OVER(PARTITION BY emp_id ORDER BY date ASC)")

In [ ]:
# EXERCISE 2: Null handling with LAG
# When using LAG(revenue, 1), the very first row in the partition will return NULL. 
# How do you default it to 0?
print("Most dialects allow a default parameter: LAG(revenue, 1, 0) OVER (...)")

In [ ]:
# EXERCISE 3: Naming Windows
# If you have 4 window functions that all use OVER(PARTITION BY region ORDER BY timestamp) 
# how can you DRY up the code?
print("Using the WINDOW clause.")
print("SELECT LAG(x) OVER w, LEAD(y) OVER w FROM tbl WINDOW w AS (PARTITION BY region ORDER BY timestamp)")

## 🎯 Summary: Key Takeaways

### What You've Learned
1. ✅ **Dual Purpose:** Window functions calculate aggregates but retain row-level granularity.
2. ✅ **The syntax:** `Function()` + `OVER` + `(PARTITION ... ORDER ... FRAMING)`.
3. ✅ **Rankings:** `ROW_NUMBER` (unique, no gaps), `RANK` (ties skipped gaps), `DENSE_RANK` (ties, no gaps).
4. ✅ **Time-Series:** `LAG` and `LEAD` are your main tools for deltas.
5. ✅ **Deduplication Trick:** Extracting record `rn = 1` ordered descending by time is a critical pipeline staple.

### Interview Confidence Checklist
- [ ] Can accurately write a 7-day rolling average script.
- [ ] Explain exactly why a window function cannot be inside a `WHERE` block.
- [ ] Distinguish when to write `RANK` vs `DENSE_RANK` based on requirements.
- [ ] Recite the Citi "Deduplicating Server State" story.

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Mastering window functions officially graduates you from Analyst to Data Engineer. 🚀